# Qwen2.5-0.5B answers and all-layer embeddings (2x T4)

This notebook uses **both Kaggle T4 GPUs**. It loads one Qwen2.5-0.5B-Instruct
replica on each GPU and sends separate mini-batches to the replicas concurrently.

It reads `unified_math_dataset.jsonl`, asks the model to solve every problem,
and writes answers plus all-layer prompt representations. The output includes
mean-pooled prompt embeddings and the final non-padding prompt-token embedding
for every hidden-state layer (including the input embedding output). Embeddings
are saved in shards to keep RAM use bounded.

The notebook requires a Kaggle GPU runtime with two visible CUDA devices.


In [1]:
# Install these if they are missing in the notebook environment.
%pip install -q -U transformers accelerate sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.1 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import shutil
import tempfile
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
GPU_COUNT = 2
BATCH_SIZE_PER_GPU = 16
GLOBAL_BATCH_SIZE = GPU_COUNT * BATCH_SIZE_PER_GPU
SHARD_SIZE = 64
MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 512
RUN_MODE = "test"  # Run the first 100 eligible rows first; change to "full" for the entire eligible dataset.
TEST_LIMIT = 100

local_candidates = [
    Path("/kaggle/input/datasets/ashbornmonarch/unified-math-dataset"),
    Path("unified_math_dataset.jsonl"),
]
DATA_PATH = next((path.resolve() for path in local_candidates if path.is_file()), None)
if DATA_PATH is None:
    kaggle_input = Path("/kaggle/input")
    kaggle_matches = sorted(kaggle_input.glob("**/unified_math_dataset.jsonl")) if kaggle_input.exists() else []
    if kaggle_matches:
        DATA_PATH = kaggle_matches[0].resolve()
if DATA_PATH is None:
    raise FileNotFoundError(
        "Cannot find unified_math_dataset.jsonl. Attach the Kaggle dataset as notebook input "
        "or place the file beside this notebook."
    )

if RUN_MODE not in {"test", "full"}:
    raise ValueError('RUN_MODE must be either "test" or "full".')
if "/kaggle/input/" in DATA_PATH.as_posix():
    SORTED_DATA_PATH = Path("/kaggle/working/unified_math_dataset_sorted_by_question_char_count.jsonl")
    OUTPUT_ROOT = Path("/kaggle/working")
else:
    SORTED_DATA_PATH = DATA_PATH
    OUTPUT_ROOT = DATA_PATH.parent
OUTPUT_NAME = (
    "qwen2_5_0_5b_unified_output_test100_512"
    if RUN_MODE == "test"
    else "qwen2_5_0_5b_unified_output_full_512"
)
OUTPUT_DIR = OUTPUT_ROOT / OUTPUT_NAME

PROMPT_TEMPLATE = """Solve this math problem. Show brief reasoning, then finish with:
Final answer: <answer>

Problem:
{question}
"""

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Unified dataset not found: {DATA_PATH}")
if torch.cuda.device_count() < GPU_COUNT:
    raise RuntimeError(f"Expected {GPU_COUNT} visible CUDA GPUs, found {torch.cuda.device_count()}.")

print(f"Input: {DATA_PATH}")
print(f"Sorted dataset output: {SORTED_DATA_PATH}")
print(f"Run mode: {RUN_MODE}")
print(f"Output: {OUTPUT_DIR}")
for gpu_index in range(GPU_COUNT):
    print(f"GPU {gpu_index}: {torch.cuda.get_device_name(gpu_index)}")
print(f"Effective batch size: {GLOBAL_BATCH_SIZE} ({BATCH_SIZE_PER_GPU} per GPU)")
print(f"Input prompt limit: {MAX_INPUT_TOKENS} tokens; longer prompts will be excluded.")


Input: /kaggle/input/datasets/ashbornmonarch/unified-math-dataset/unified_math_dataset.jsonl
Output: /kaggle/working/qwen2_5_0_5b_unified_output_4481_onward_512
GPU 0: Tesla T4
GPU 1: Tesla T4
Effective batch size: 32 (16 per GPU)
Input prompt limit: 512 tokens; longer prompts will be excluded.


In [3]:
source_rows = []
with DATA_PATH.open("r", encoding="utf-8") as stream:
    for line in stream:
        if line.strip():
            source_rows.append(json.loads(line))

# Sort by question character count (descending), as requested. Equal lengths keep their input order.
for row in source_rows:
    row["question_char_count"] = len(row["question"])
all_rows = sorted(source_rows, key=lambda row: row["question_char_count"], reverse=True)

# Write a sorted copy beside Kaggle's read-only input, or replace the local unified JSONL atomically.
SORTED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(
    mode="w", encoding="utf-8", newline="\n", delete=False,
    dir=SORTED_DATA_PATH.parent, suffix=".tmp",
) as sorted_stream:
    sorted_temp_path = Path(sorted_stream.name)
    for row in all_rows:
        sorted_stream.write(json.dumps(row, ensure_ascii=False) + "\n")
os.replace(sorted_temp_path, SORTED_DATA_PATH)

# Apply the model's actual chat template to count input tokens; this filter is separate from character sorting.
length_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
PROMPT_TOKEN_COUNTS = {}
for start in tqdm(range(0, len(all_rows), 256), desc="Counting Qwen prompt tokens"):
    chunk = all_rows[start:start + 256]
    prompts = [
        length_tokenizer.apply_chat_template(
            [{"role": "user", "content": PROMPT_TEMPLATE.format(question=row["question"])}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for row in chunk
    ]
    encoded = length_tokenizer(prompts, add_special_tokens=True, truncation=False, padding=False)
    for row, input_ids in zip(chunk, encoded["input_ids"]):
        PROMPT_TOKEN_COUNTS[row["example_id"]] = len(input_ids)
del length_tokenizer

eligible_rows = [
    row for row in all_rows
    if PROMPT_TOKEN_COUNTS[row["example_id"]] <= MAX_INPUT_TOKENS
]
excluded_over_limit = len(all_rows) - len(eligible_rows)
if RUN_MODE == "test":
    selected_rows = eligible_rows[:TEST_LIMIT]
else:
    selected_rows = eligible_rows
rows = selected_rows

print(f"Raw dataset records: {len(all_rows):,}")
print(f"Sorted by question characters: {all_rows[0]['question_char_count']:,} down to {all_rows[-1]['question_char_count']:,}")
print(f"Prompts over {MAX_INPUT_TOKENS} tokens excluded: {excluded_over_limit:,}")
print(f"Eligible records: {len(eligible_rows):,}")
print(f"Selected for {RUN_MODE}: {len(selected_rows):,}")
print("Selected splits:", dict(Counter(row["split"] for row in selected_rows)))


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Counting prompt tokens:   0%|          | 0/66 [00:00<?, ?it/s]

Raw dataset records: 16,792
Continuation starts at raw record 4,481.
Remaining rows over 512 prompt tokens (excluded): 80
Eligible remaining rows: 12,232
Eligible remaining splits: {'train': 10416, 'test': 1816}


In [4]:
RUNNERS = []
for gpu_index in range(GPU_COUNT):
    device = torch.device(f"cuda:{gpu_index}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    ).to(device)
    model.eval()
    RUNNERS.append({"model": model, "tokenizer": tokenizer, "device": device})
    print(f"Replica {gpu_index} loaded on {device}; blocks={model.config.num_hidden_layers}")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Replica 0 loaded on cuda:0; blocks=24


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Replica 1 loaded on cuda:1; blocks=24


In [5]:
@torch.inference_mode()
def answer_and_embed(batch_rows, runner):
    torch.cuda.set_device(runner["device"])
    tokenizer = runner["tokenizer"]
    model = runner["model"]
    device = runner["device"]

    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": PROMPT_TEMPLATE.format(question=row["question"])}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for row in batch_rows
    ]
    encoded = tokenizer(
        prompts,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    if encoded["input_ids"].shape[1] > MAX_INPUT_TOKENS:
        raise ValueError("A prompt exceeded the 512-token filter after batching; refusing to truncate it.")
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    # Capture the embedding output and the output of every transformer block.
    forward = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        use_cache=False,
    )
    if forward.hidden_states is None:
        raise RuntimeError("Model did not return hidden states.")
    layers = torch.stack(forward.hidden_states, dim=1)  # [batch, layers, seq, hidden]
    mask = attention_mask[:, None, :, None].to(layers.dtype)
    mean_layers = (layers * mask).sum(dim=2) / mask.sum(dim=2).clamp_min(1)
    last_prompt_layers = layers[:, :, -1, :]  # left padding: final position is the last real token
    del forward, layers, mask

    generated = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )
    answer_ids = generated[:, input_ids.shape[1]:]
    answers = tokenizer.batch_decode(answer_ids, skip_special_tokens=True)
    del generated, answer_ids, input_ids, attention_mask, encoded

    return (
        [answer.strip() for answer in answers],
        mean_layers.cpu().half(),
        last_prompt_layers.cpu().half(),
    )


In [6]:
# Save either the 100-row smoke run or the full eligible dataset in its own folder.
metadata_path = OUTPUT_DIR / "metadata.jsonl"
manifest_path = OUTPUT_DIR / "manifest.json"
selected_ids = {row["example_id"] for row in selected_rows}


def read_jsonl(path):
    with path.open("r", encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def compatible_manifest(manifest):
    return (
        manifest.get("model") == MODEL_ID
        and manifest.get("run_mode") == RUN_MODE
        and manifest.get("max_input_tokens") == MAX_INPUT_TOKENS
        and manifest.get("selected_records") == len(selected_rows)
        and manifest.get("sort_order") == "question_char_count_desc"
    )


def find_or_restore_partial_output():
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        if metadata_path.is_file() and manifest_path.is_file():
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            if compatible_manifest(manifest):
                return True
        raise FileExistsError(
            f"{OUTPUT_DIR} contains files from a different run. "
            "Preserve them and choose a new output folder before retrying."
        )

    # If a Kaggle session restarted, an attached partial output for this same mode can be resumed.
    kaggle_input = Path("/kaggle/input")
    candidates = list(kaggle_input.glob("**/manifest.json")) if kaggle_input.exists() else []
    valid = []
    for manifest_file in candidates:
        try:
            manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if compatible_manifest(manifest):
            valid.append((manifest.get("total_records", 0), manifest_file.parent))
    if valid:
        _, source_dir = max(valid, key=lambda item: item[0])
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source_dir, OUTPUT_DIR, dirs_exist_ok=True)
        return True

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    return False


resuming_run = find_or_restore_partial_output()
if resuming_run:
    previous_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    previous_metadata = read_jsonl(metadata_path)
    if len(previous_metadata) != previous_manifest.get("total_records"):
        raise ValueError("Output metadata row count does not match its manifest.")
    if len({row["example_id"] for row in previous_metadata}) != len(previous_metadata):
        raise ValueError("Output contains duplicate example IDs.")
    completed_ids = {row["example_id"] for row in previous_metadata}
    if not completed_ids.issubset(selected_ids):
        raise ValueError("Output contains IDs outside the selected dataset rows.")
else:
    previous_manifest = {}
    previous_metadata = []
    completed_ids = set()

rows = [row for row in selected_rows if row["example_id"] not in completed_ids]
print(f"Completed rows already saved: {len(previous_metadata):,}")
print(f"Rows left in this run: {len(rows):,}")

pending_rows = []
pending_mean = []
pending_last = []
shard_summaries = list(previous_manifest.get("shards", []))
split_counts = Counter(row["split"] for row in previous_metadata)
total_written = len(previous_metadata)
processed_this_run = 0
shard_index = 0
if shard_summaries:
    shard_index = max(int(item["file"].split("_")[-1].split(".")[0]) for item in shard_summaries) + 1


def save_manifest():
    manifest = {
        "model": MODEL_ID,
        "run_mode": RUN_MODE,
        "sort_order": "question_char_count_desc",
        "sorted_dataset_path": str(SORTED_DATA_PATH),
        "raw_dataset_records": len(all_rows),
        "eligible_records_at_or_below_token_limit": len(eligible_rows),
        "selected_records": len(selected_rows),
        "total_records": total_written,
        "split_counts": dict(split_counts),
        "replicas": GPU_COUNT,
        "devices": [f"cuda:{i}" for i in range(GPU_COUNT)],
        "batch_size_per_gpu": BATCH_SIZE_PER_GPU,
        "effective_batch_size": GLOBAL_BATCH_SIZE,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "excluded_over_token_limit": excluded_over_limit,
        "embedding_definition": (
            "For each prompt: mean-pooled non-padding token representation and final "
            "non-padding token representation at every hidden-state layer, including "
            "the input embedding output. Generated answer tokens are not included in embeddings."
        ),
        "shards": shard_summaries,
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def flush_shard(metadata_stream):
    global shard_index, total_written
    if not pending_rows:
        return
    shard_name = f"embeddings_{shard_index:05d}.pt"
    payload = {
        "example_ids": [row["example_id"] for row in pending_rows],
        "layer_embeddings": torch.stack(pending_mean).contiguous(),
        "layer_last_token_embeddings": torch.stack(pending_last).contiguous(),
    }
    torch.save(payload, OUTPUT_DIR / shard_name)
    for row_number, row in enumerate(pending_rows):
        row["prompt_tokens"] = PROMPT_TOKEN_COUNTS[row["example_id"]]
        row["embedding_shard"] = shard_name
        row["embedding_row"] = row_number
        metadata_stream.write(json.dumps(row, ensure_ascii=False) + "\n")
    metadata_stream.flush()
    shard_summaries.append({
        "file": shard_name,
        "count": len(pending_rows),
        "layer_embedding_shape": list(payload["layer_embeddings"].shape[1:]),
        "dtype": str(payload["layer_embeddings"].dtype),
    })
    total_written += len(pending_rows)
    pending_rows.clear()
    pending_mean.clear()
    pending_last.clear()
    shard_index += 1
    save_manifest()


if not resuming_run:
    save_manifest()  # Keep a valid checkpoint even if the first inference batch fails.
metadata_mode = "a" if resuming_run else "w"
with metadata_path.open(metadata_mode, encoding="utf-8", newline="\n") as metadata_stream:
    with ThreadPoolExecutor(max_workers=GPU_COUNT) as executor:
        batch_starts = range(0, len(rows), GLOBAL_BATCH_SIZE)
        for start in tqdm(batch_starts, desc=f"Qwen2.5-0.5B {RUN_MODE} run on 2 GPUs"):
            global_batch = rows[start:start + GLOBAL_BATCH_SIZE]
            jobs = []
            for gpu_index in range(GPU_COUNT):
                lower = gpu_index * BATCH_SIZE_PER_GPU
                upper = lower + BATCH_SIZE_PER_GPU
                chunk = global_batch[lower:upper]
                if chunk:
                    future = executor.submit(answer_and_embed, chunk, RUNNERS[gpu_index])
                    jobs.append((chunk, future))

            for chunk, future in jobs:
                answers, mean_embeddings, last_embeddings = future.result()
                for source_row, answer, mean_embedding, last_embedding in zip(
                    chunk, answers, mean_embeddings, last_embeddings
                ):
                    result = dict(source_row)
                    result["qwen_model"] = MODEL_ID
                    result["qwen_response"] = answer
                    existing_responses = [
                        item for item in source_row.get("responses", [])
                        if item.get("model") != MODEL_ID
                    ]
                    result["responses"] = existing_responses + [{
                        "model": MODEL_ID,
                        "text": answer,
                    }]
                    pending_rows.append(result)
                    pending_mean.append(mean_embedding)
                    pending_last.append(last_embedding)
                    split_counts[source_row["split"]] += 1
                    processed_this_run += 1
                    if len(pending_rows) >= SHARD_SIZE:
                        flush_shard(metadata_stream)
                del answers, mean_embeddings, last_embeddings

            for gpu_index in range(GPU_COUNT):
                with torch.cuda.device(gpu_index):
                    torch.cuda.empty_cache()

    flush_shard(metadata_stream)

save_manifest()
print(json.dumps({
    "output_dir": str(OUTPUT_DIR),
    "run_mode": RUN_MODE,
    "records_written": total_written,
    "records_processed_this_run": processed_this_run,
    "excluded_over_512_tokens": excluded_over_limit,
    "shards": len(shard_summaries),
}, indent=2))


Output: /kaggle/working/qwen2_5_0_5b_unified_output_4481_onward_512
Continuation rows already complete: 0
Eligible continuation rows left: 12,232
Rows over 512 prompt tokens excluded: 80


Qwen2.5-0.5B continuation on 2 GPUs:   0%|          | 0/383 [00:00<?, ?it/s]

: 

## Dataset ordering and run modes

- The unified JSONL is sorted by `question_char_count` in descending order. Equal character counts retain their existing order.
- The tokenizer separately counts the complete Qwen prompt. Prompts over 512 tokens are excluded; accepted prompts are not truncated.
- The default `RUN_MODE = "test"` generates answers and embeddings for the first 100 eligible rows in character-sorted order. This smoke run writes to `/kaggle/working/qwen2_5_0_5b_unified_output_test100_512`.
- After reviewing the test output, set `RUN_MODE = "full"` and run again. Full mode processes all eligible rows from the sorted unified dataset and writes to `/kaggle/working/qwen2_5_0_5b_unified_output_full_512`.
- The full output covers the full sorted dataset subject to the 512-token input cap; it runs from the beginning and does not depend on the earlier 4,480-row output.
